# AI-Publisher GHCR Image Upload

Drive'daki Docker imaj arsivlerini (`.tar.gz`) GHCR'a (GitHub Container Registry) yukler.

### Akis:
1. Google Drive baglanir
2. `build_all_v2.sh` ile olusturulmus `.tar.gz` dosyalari taranir
3. Podman kurulur (daemonless, Colab'da calisir)
4. GHCR'a giris yapilir (GitHub PAT gerekli)
5. Her arsiv ayri ayri GHCR'a push edilir

**Gereken Colab Secret:** `GITHUB_PAT` (veya `GITHUB_TOKEN`, `GH_TOKEN`)

---
Her hucreyi sirayla calistirin.

## Hucre 1: Google Drive Baglantisi

In [ ]:
import os
import subprocess
import sys
import glob
from pathlib import Path

# Google Drive mount
from google.colab import drive
drive.mount('/content/drive')

# Konfigurasyon
DRIVE_IMAGE_DIR = "/content/drive/MyDrive/Colab Notebooks/docker/images"
GHCR_NAMESPACE = "ghcr.io/arda-avci"
IMAGE_PREFIX = "ai-publisher-"  # tar.gz icindeki image name prefix

print(f"Image dizini: {DRIVE_IMAGE_DIR}")
print(f"GHCR hedef: {GHCR_NAMESPACE}")

## Hucre 2: Mevcut Arsivleri Tara

In [ ]:
image_dir = Path(DRIVE_IMAGE_DIR)
if not image_dir.exists():
    print(f"HATA: {DRIVE_IMAGE_DIR} bulunamadi! Drive mount oldugunu dogrulayin.")
    sys.exit(1)

archives = sorted(image_dir.glob("*.tar.gz"))
print(f"\nBulunan arsiv sayisi: {len(archives)}\n")

for a in archives:
    size_mb = a.stat().st_size / (1024 * 1024)
    print(f"  {a.name}  ({size_mb:.0f} MB)")

if not archives:
    print("\nArsiv bulunamadi!")
    print(f"Beklenen dizin: {DRIVE_IMAGE_DIR}")
    print("\nOnce build_all_v2.sh calistirin.")

## Hucre 3: Podman Kurulumu

Podman daemonless container engine'dir. Docker imajlarini Docker daemon olmadan load/push edebilir.

In [ ]:
def run(cmd, check=True, capture=False):
    """Shell komutu calistir. Her zaman result objesi doner."""
    import subprocess
    result = subprocess.run(cmd, shell=True, text=True,
                           capture_output=capture)
    if capture:
        result._stdout = result.stdout.strip() if result.stdout else ""
    if check and result.returncode != 0:
        print(f"[HATA] Komut basarisiz: {cmd}")
        print(f"STDERR: {result.stderr}")
        sys.exit(1)
    return result

# Podman kurulu mu?
r = run("which podman", check=False, capture=True)
if r.returncode == 0:
    print(f"Podman mevcut: {r._stdout}")
    print(run("podman --version", capture=True)._stdout)
else:
    print("Podman kuruluyor...")
    run("apt-get update -qq")
    run("apt-get install -y -qq podman")
    print("Podman kuruldu.")
    print(run("podman --version", capture=True)._stdout)

## Hucre 4: GHCR Giris

Colab Secret'tan GitHub PAT okuyup GHCR'a giris yapar.
PAT'in `write:packages` izni olmali.

In [ ]:
from google.colab import userdata
import base64, os, json, requests, sys, traceback

gh_token = None
_token_errors = []

_all_keys = ["GITHUB_PAT", "GITHUB_TOKEN", "GH_TOKEN", "github_token", "GITHUB_TOKEN_1"]
for key in _all_keys:
    try:
        gh_token = userdata.get(key)
        if gh_token:
            print(f"Token bulundu: {key} (ilk 8: {gh_token[:8]}...)")
            break
        else:
            _token_errors.append(f"{key} = bos")
    except Exception as e:
        _token_errors.append(f"{key} -> {type(e).__name__}")

if not gh_token:
    print("HATA: GitHub PAT bulunamadi!")
    if _token_errors:
        for err in _token_errors: print(f"  {err}")
    print("\nColab > Kilit ikonu > Secrets > GITHUB_TOKEN ekle")
    sys.exit(1)

# Strip Windows newlines from token
gh_token_val = gh_token.strip()
ghcr_username = "arda-avci"

# Podman auth.json
_auth_b64 = base64.b64encode(f"{ghcr_username}:{gh_token_val}".encode()).decode()
_auth_dir = os.path.expanduser("~/.config/containers")
os.makedirs(_auth_dir, exist_ok=True)
with open(os.path.join(_auth_dir, "auth.json"), "w") as _f:
    json.dump({"auths": {"ghcr.io": {"auth": _auth_b64}}}, _f)
print(f"Auth dosyasi yazildi")

GHCR_AUTH = f"{ghcr_username}:{gh_token_val}"
os.environ["CRANE_AUTH"] = GHCR_AUTH

# Token test
_gh_headers = {"Authorization": f"Bearer {gh_token_val}", "Accept": "application/vnd.github.v3+json"}
_r = requests.get("https://api.github.com/user", headers=_gh_headers)
if _r.status_code == 200:
    _login = _r.json()["login"]
    _scopes = _r.headers.get("X-OAuth-Scopes", "").split(", ")
    print(f"[OK] Token gecerli. GitHub: {_login}")
    print(f"[SCOPE] {', '.join(_scopes)}")
    if not any(s in _scopes for s in ["write:packages", "repo"]):
        print("[!] DIKKAT: write:packages yok!")
else:
    print(f"[FAIL] Token gecersiz! HTTP {_r.status_code}")
    sys.exit(1)

## Hucre 5: Imajlari GHCR'a Push Et

Her `.tar.gz` dosyasini sirayla Podman'a yukler ve GHCR'a push eder.

Not: Buyuk imajlar (video modelleri 5-15 GB) push suresini uzatabilir.

In [ ]:
import time, shutil, shlex, pathlib, requests

TEMP_DIR = pathlib.Path("/content/tmp_images")
TEMP_DIR.mkdir(parents=True, exist_ok=True)

def run_cap(cmd, timeout=None):
    import subprocess
    kwargs = dict(shell=True, text=True, capture_output=True)
    if timeout: kwargs["timeout"] = timeout
    try:
        r = subprocess.run(cmd, **kwargs)
    except subprocess.TimeoutExpired:
        class R: returncode = 124; stdout = ""; stderr = ""; _stdout = ""
        return R()
    r._stdout = r.stdout.strip() if r.stdout else ""
    return r

# Skopeo kur
if not shutil.which("skopeo"):
    print("Skopeo kuruluyor...")
    run_cap("apt-get install -y -qq skopeo")
    if shutil.which("skopeo"): print("Skopeo kuruldu.")

# Crane kur
crane_path = shutil.which("crane")
if not crane_path:
    print("Crane kuruluyor...")
    r = run_cap(
        "wget -q https://github.com/google/go-containerregistry/releases/latest/download/"
        "go-containerregistry_Linux_x86_64.tar.gz -O /tmp/crane.tar.gz && "
        "tar -xzf /tmp/crane.tar.gz -C /usr/local/bin/ crane && chmod +x /usr/local/bin/crane",
        timeout=120)
    if r.returncode == 0: print("Crane kuruldu.")
    crane_path = shutil.which("crane")

print(f"\nImaj push islemi basliyor... ({len(archives)} imaj)\n")

total = len(archives)
success = 0
failed = []

for idx, archive_path in enumerate(archives, 1):
    model_name = archive_path.stem.replace(".tar", "")
    ghcr_image = f"{GHCR_NAMESPACE}/{IMAGE_PREFIX}{model_name}:latest"
    size_mb = archive_path.stat().st_size / (1024 * 1024)

    print(f"[{idx}/{total}] {model_name}  ({size_mb:.0f} MB)")
    print(f"  -> {ghcr_image}")

    start = time.time()
    temp_tar = TEMP_DIR / f"{model_name}.tar"
    q_arc = shlex.quote(str(archive_path))
    q_tar = shlex.quote(str(temp_tar))

    # 1) Decompress
    print(f"  [1/4] Decompress...")
    r = run_cap(f"gunzip -c {q_arc} > {q_tar}")
    if r.returncode != 0:
        print(f"  [HATA] Decompress: {r.stderr[:200]}")
        failed.append(model_name); continue
    print(f"  [OK] {temp_tar.stat().st_size/(1024*1024):.0f} MB")

    pushed = False

    # 2a) Podman load + push
    print(f"  [2/4] Podman load...")
    r = run_cap(f"podman load -i {q_tar}")
    if r.returncode == 0:
        loaded_name = None
        for line in r._stdout.split("\n"):
            if "Loaded image" in line:
                parts = line.split(":", 1)
                loaded_name = parts[-1].strip()
                if "sha256:" in loaded_name: loaded_name = None
                break
        if not loaded_name: loaded_name = f"localhost/{IMAGE_PREFIX}{model_name}:local"
        print(f"  Load: {loaded_name}")

        for attempt in range(3):
            print(f"  [3/4] Podman push (deneme {attempt+1}/3)...")
            r = run_cap(
                f"podman tag {loaded_name} {ghcr_image} && podman push {ghcr_image}",
                timeout=1800)
            if r.returncode == 0:
                elapsed = time.time() - start
                print(f"  [OK] Push tamam. ({elapsed:.0f}s)")
                pushed = True
                break
            print(f"  [WARN] Basarisiz: {r.stderr[-200:]}")
            if attempt < 2: time.sleep(15)
        run_cap(f"podman rmi {loaded_name} 2>/dev/null")
        run_cap(f"podman rmi {ghcr_image} 2>/dev/null")

    if pushed:
        temp_tar.unlink(missing_ok=True)
        success += 1; continue

    # 2b) Skopeo copy
    if shutil.which("skopeo"):
        print(f"  [2/4] Skopeo copy...")
        r = run_cap(
            f"skopeo copy --dest-creds={GHCR_AUTH} "
            f"docker-archive:{q_tar} docker://{ghcr_image}",
            timeout=1800)
        if r.returncode == 0:
            print(f"  [OK] Skopeo push tamam. ({time.time()-start:.0f}s)")
            temp_tar.unlink(missing_ok=True); success += 1; continue
        print(f"  [WARN] Skopeo: {r.stderr[-200:]}")

    # 2c) Crane push
    if crane_path:
        print(f"  [2/4] Crane push...")
        r = run_cap(f"crane push {q_tar} {ghcr_image}", timeout=1800)
        if r.returncode == 0:
            print(f"  [OK] Crane push tamam. ({time.time()-start:.0f}s)")
            temp_tar.unlink(missing_ok=True); success += 1; continue
        print(f"  [WARN] Crane: {r.stderr[-200:]}")

    print(f"  [FAIL] {model_name} — tum yontemler basarisiz!")
    failed.append(model_name)
    temp_tar.unlink(missing_ok=True)

print(f"\n{'='*60}")
print(f"SONUC: {success}/{total} basarili")
if failed: print(f"BASARISIZ: {failed}")
print("="*60)

## Hucre 6: Ozet

In [ ]:
print("\n" + "=" * 60)
print("SONUC")
print("=" * 60)
print(f"  Basarili: {success}/{total}")

if failed:
    print(f"\n  Basarisiz ({len(failed)}):")
    for f in failed:
        print(f"    - {f}")
else:
    print("\n  Tum imajlar GHCR'a yuklendi.")

print("\nDogrulama:")
print(f"  https://github.com/Arda-Avci/AI-Publisher/pkgs/container/")
print(f"  veya: https://github.com/settings/packages")

## Hucre 7: (Opsiyonel) Dogrulama

GHCR'a push edilen imajlari Podman ile cekip listeler.

In [ ]:
import json

test_model = archives[0].stem if archives else "cogvideox"
if test_model.endswith(".tar"):
    test_model = test_model[:-4]
test_image = f"{GHCR_NAMESPACE}/{IMAGE_PREFIX}{test_model}:latest"

print(f"Test imaji cekiliyor: {test_image}")
result = run(f"podman pull {test_image}", check=False)
if result.returncode == 0:
    print(f"[OK] {test_image} basariyla cekildi.")
    run(f"podman rmi {test_image} 2>/dev/null", check=False)
else:
    print(f"[WARN] Pull basarisiz. Image GHCR'da goruntulenemiyor olabilir.")
    print("Gorunurluk: GitHub > Packages > package > Package settings > Change visibility > Public")

---

### Sorun Giderme

| Sorun | Cozum |
|-------|-------|
| `podman: command not found` | Hucre 3 podman kurulumu basarisiz. `apt-cache search podman` ile paket adini dogrulayin. |
| `denied: permission` | GitHub PAT'in `write:packages` izni yok. PAT'i guncelleyin. |
| `unauthorized` | Token gecersiz veya süresi dolmus. Yeni PAT olusturun. |
| Disk dolu | Colab Runtime > Disconnect and delete runtime > Yeniden baslatip sadece ghcr upload yapin. |
| `tar.gz` dosyasi bozuk | Dosyayi Drive'dan silip `build_all_v2.sh` ile yeniden olusturun. |
| Image cekilemiyor | GitHub > Packages > `Arda-Avci/ai-publisher-{model}` > **Make public** |